# Experiment 7 - Part 3: Open-Source Vision Models (SmolVLM locally, LLaVA on Colab)

**Course:** HAAI302COM - Artificial Intelligence Laboratory

Gemini is an API: we cannot see or change the model. Open-source vision-language models (VLMs) run on our own machine. This notebook has two halves:

1. **SmolVLM-256M**, a tiny model (256 million parameters) that runs on a laptop CPU. We take it apart, run it on our 90 questions and compare it with Gemini.
2. **LLaVA-1.5-7B**, the model named in the syllabus. It needs a GPU, so its code is written out for **Google Colab** and guarded so that it is skipped on a machine without one.

The comparison is the real lesson: what do we gain and lose when we move from a large hosted model to a small local one?

## 0. Setup

We reuse the question set and the scoring functions from the earlier notebooks, and the cached Gemini answers (run notebook 02 first, or the Gemini column will make fresh API calls).

In [1]:
import json, time, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import torch
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText
import vqa_utils as vu

pd.set_option("display.max_colwidth", 70)
pd.set_option("display.width", 170)
df = pd.DataFrame(json.load(open("data/vqa_set.json")))
torch.set_num_threads(8)
print("torch", torch.__version__, "| GPU available:", torch.cuda.is_available())

torch 2.14.0+cu130 | GPU available: False


## 1. Load SmolVLM and look inside

A VLM is three parts glued together: a **vision encoder** that turns image patches into vectors, a **connector** that maps those vectors into the language model's input space, and a **language model** that reads image-vectors and question tokens together and writes the answer. `AutoProcessor` prepares both inputs (image resizing and text tokenising); `AutoModelForImageTextToText` is the network. The first run downloads about 0.5 GB.

In [2]:
MID = "HuggingFaceTB/SmolVLM-256M-Instruct"
proc = AutoProcessor.from_pretrained(MID)
model = AutoModelForImageTextToText.from_pretrained(MID, dtype=torch.float32).eval()
print("parameters:", round(sum(p.numel() for p in model.parameters()) / 1e6), "million")

[transformers] Model config: pad_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 128002. This may result in unexpected behavior.


Loading weights:   0%|          | 0/471 [00:00<?, ?it/s]

parameters: 256 million


In [3]:
# count parameters per top-level block: encoder, connector, language model
parts = {}
for name, p in model.named_parameters():
    top = ".".join(name.split(".")[:2])
    parts[top] = parts.get(top, 0) + p.numel()
pd.Series(parts).div(1e6).round(1).rename("million params").to_frame()

,million params
model.vision_model,86.4
model.connector,7.1
model.text_model,134.6
lm_head.weight,28.4


**What to notice:** the language model (134.6M text model plus 28.4M output layer) is about 60 percent of the network, the vision encoder (86.4M) is about a third, and the connector that links them is only 7.1M, 3 percent. The connector is small because it only has to translate between two spaces that are already trained; in LLaVA-style training, it is the first part trained.

## 2. From image to tokens: image splitting

SmolVLM cuts a large image into several 512 by 512 tiles (so that small text stays readable) and adds one downsized overview tile. Each tile becomes dozens of tokens. `do_image_splitting` turns this on or off, so it is the speed-versus-detail dial of the model. We build the model input for one document image both ways and count.

In [4]:
row = df[df.kind == "document"].iloc[0]
img = Image.open(row.image)
msgs = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": row.question}]}]
prompt = proc.apply_chat_template(msgs, add_generation_prompt=True)
for split in [True, False]:
    inp = proc(text=prompt, images=[img], return_tensors="pt", do_image_splitting=split)
    print(f"splitting={split!s:5}  tiles={inp.pixel_values.shape[1]:2d}  total input tokens={inp.input_ids.shape[1]}")

splitting=True   tiles=17  total input tokens=1158
splitting=False  tiles= 1  total input tokens=98


**What to notice:** splitting turns one document image into 17 tiles and 1,158 input tokens, while the unsplit version uses a single tile and 98 tokens, about 12 times fewer. The language model must process every one of those tokens, so splitting costs time roughly in proportion. The price of speed is detail: one 512 pixel tile cannot hold a full page of print.

## 3. One question, step by step

We wrap generation into `answer()`: build the chat prompt with an `<image>` slot, let the processor fill it, call `model.generate` with `do_sample=False` (greedy decoding, so the answer is repeatable) and decode only the **new** tokens. We use the same short-answer instruction as with Gemini and turn splitting off for speed.

In [5]:
INSTR = " Answer with a single word, number or short phrase."

def answer(path, question, split=False, max_new_tokens=24):
    img = Image.open(path).convert("RGB")
    msgs = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": question + INSTR}]}]
    inp = proc(text=proc.apply_chat_template(msgs, add_generation_prompt=True), images=[img],
               return_tensors="pt", do_image_splitting=split)
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=max_new_tokens, do_sample=False)
    return proc.batch_decode(out[:, inp.input_ids.shape[1]:], skip_special_tokens=True)[0].strip()

t = time.time(); r = df[df.kind == "photo"].iloc[2]
print(r.question, "->", answer(r.image, r.question), "| reference:", r.gold, "|", round(time.time() - t, 1), "s")

What color is the court? -> Green. | reference: green | 4.2 s


**What to notice:** the model answers 'Green.' in 4.2 seconds on CPU, correct apart from capitalisation and a full stop, which our metric normalises away. Four seconds for a single short answer is the baseline cost of running a model locally without a GPU.

## 4. Run all 90 questions

The loop saves every answer to `smolvlm_answers.json` as it goes, so an interrupted run resumes where it stopped. Without image splitting each question takes a few seconds on CPU.

In [6]:
import os
CACHE = "smolvlm_answers.json"
cache = json.load(open(CACHE)) if os.path.exists(CACHE) else {}
t0 = time.time()
for i, r in df.iterrows():
    if str(i) not in cache:
        cache[str(i)] = answer(r.image, r.question)
        json.dump(cache, open(CACHE, "w"))
df["small"] = [cache[str(i)] for i in df.index]
print("done in", round(time.time() - t0), "s (0 means everything was cached)")

done in 403 s (0 means everything was cached)


**What to notice:** the full run of 90 questions took 403 seconds, about 4.5 seconds each. Because we saved answers as we went, re-running costs nothing. On a GPU the same loop would take a few minutes.

## 5. Score SmolVLM against Gemini

The Gemini answers come from `vqa_utils.ask()` with the same prompt as notebook 02, so they are read from the cache. Both models are scored with the same metric per image type.

In [7]:
SYS = "You answer questions about images. Reply with only the answer: one word, one number or a short phrase. No sentences."
df["gemini"] = [vu.ask(r.image, r.question, system=SYS)["answer"] for _, r in df.iterrows()]
df["s_small"] = [vu.score(r.kind, r.small, r.answers, r.gold) for _, r in df.iterrows()]
df["s_gemini"] = [vu.score(r.kind, r.gemini, r.answers, r.gold) for _, r in df.iterrows()]
df.groupby("kind")[["s_small", "s_gemini"]].mean().round(3).rename(columns={"s_small": "SmolVLM-256M", "s_gemini": "Gemini flash-lite"})

,SmolVLM-256M,Gemini flash-lite
kind,,
chart,0.200,0.800
document,0.188,0.816
photo,0.367,0.633


**What to notice:** SmolVLM reaches 0.37 on photos, 0.20 on charts and 0.19 on documents, against Gemini's 0.63, 0.80 and 0.82. On photos it keeps more than half of Gemini's score. On charts and documents it keeps about a quarter. Size and resolution matter most for tasks that need reading: a 256M model can name a colour but cannot reliably read a number off a bar.

## 6. What the small model gets wrong

We read the SmolVLM answers next to the references for questions where Gemini was right. These are the cases the extra size bought.

In [8]:
lost = df[(df.s_gemini >= .5) & (df.s_small < .5)][["kind", "question", "gold", "small", "gemini"]].copy()
lost["question"] = lost.question.str[:50]
print(len(lost), "questions Gemini got right and SmolVLM got wrong")
lost.groupby("kind").head(4).sort_values("kind").reset_index(drop=True)

47 questions Gemini got right and SmolVLM got wrong


,kind,question,gold,small,gemini
0,chart,What is the percentage of people who are dissatisf,68,The Spanish public is divided on the state of democracy. A 32% maj...,68%
1,chart,Which country has the highest percentage value of,Kenya,"The U.S. is the world’s largest democracy, with a median of 63% of...",Kenya
2,chart,"The value is 40.49, find the category?",Kidney cancer,Burden.,Kidney cancer
3,chart,What group has the highest death rate?,70+ years old,75.7.,70+ years old
4,document,"What is the ‘actual’ value per 1000, during the ye",0.28,9.3,0.24
5,document,What is the name of the fashion wear/clothing adve,wills lifestyle,ITC,Wills Lifestyle
6,document,What is the no. of Ordinary shares held by Y. C. D,"24,26,435",1000000,"24,26,435"
7,document,What is the name of ITC's matches brand?,Aim,"ITC's matches brand is ""Powering Economic Growth"".",Aim
8,photo,What is the guy pointing at?,pots,Pans.,Pots
9,photo,Is this woman's legs stuck?,no,Yes.,No


**What to notice:** 47 of 90 questions went to Gemini only. The small model fails in two ways. First, it does not follow the format: for chart questions it wrote sentences about the topic of the chart ('The Spanish public is divided...') that ignore the question. Second, it reads the wrong thing: '9.3' for 0.28, 'ITC' for a brand name, '1000000' for a share count. Both are expected for a model that sees a whole page as a single 512 pixel tile.

## 7. Parameter experiment: image splitting on documents

Splitting costs time. Does it buy accuracy on text-heavy images? We run 8 document questions with splitting on and off and compare score and time.

In [9]:
docs = df[df.kind == "document"].iloc[:8]
rows = []
for split in [False, True]:
    t = time.time()
    preds = [answer(r.image, r.question, split=split) for _, r in docs.iterrows()]
    sc = np.mean([vu.score("document", p, r.answers, r.gold) for p, (_, r) in zip(preds, docs.iterrows())])
    rows.append(dict(image_splitting=split, anls=round(sc, 3), sec_per_question=round((time.time() - t) / len(docs), 1)))
pd.DataFrame(rows)

,image_splitting,anls,sec_per_question
0,False,0.000,4.5
1,True,0.617,43.9


**What to notice:** on 8 document questions, turning splitting on lifted the ANLS score from 0.00 to 0.62, but the time per question rose from 4.5 to 43.9 seconds, about ten times. So much of the document weakness above is a resolution problem, not a lack of model capability: given more tiles, the same tiny model can read. Eight questions are too few for a precise estimate, but the direction is clear. Running all 90 documents with splitting would take over an hour on this CPU, which is why a GPU matters.

## 8. LLaVA-1.5-7B on a Colab GPU (not run here)

LLaVA ("Large Language and Vision Assistant") connects a CLIP vision encoder to a Vicuna/Llama language model with a small projection layer. At 7 billion parameters it needs about 14 GB in half precision, which does not fit a free Colab T4 (15 GB) with room for activations. The standard fix is **4-bit quantization** with `bitsandbytes`, which brings it to roughly 4 to 5 GB.

To run this section: open the notebook in Colab, choose **Runtime > Change runtime type > T4 GPU**, run `pip install -q bitsandbytes accelerate`, and re-run the cells. On a machine without a CUDA GPU the guard below skips the section, so the rest of the notebook is unaffected.

LLaVA expects its own prompt format: `USER: <image>\n{question} ASSISTANT:`. The processor replaces `<image>` with 576 image tokens (a 24 by 24 grid of patches from a 336-pixel image).

In [10]:
from transformers import LlavaForConditionalGeneration, BitsAndBytesConfig
LLAVA = "llava-hf/llava-1.5-7b-hf"

def load_llava():
    # 4-bit weights, float16 compute: the settings that make a 7B model fit in a T4
    q = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4")
    p = AutoProcessor.from_pretrained(LLAVA)
    m = LlavaForConditionalGeneration.from_pretrained(LLAVA, quantization_config=q, device_map="auto")
    return p, m

if torch.cuda.is_available():
    lproc, lmodel = load_llava()
    print("LLaVA loaded, GPU memory:", round(torch.cuda.memory_allocated() / 1e9, 1), "GB")
else:
    print("No CUDA GPU here: skipping the LLaVA section (run it on Colab).")

No CUDA GPU here: skipping the LLaVA section (run it on Colab).


The generation function mirrors `answer()` above, with LLaVA's prompt format. Greedy decoding keeps it repeatable.

In [11]:
def llava_answer(path, question, max_new_tokens=24):
    img = Image.open(path).convert("RGB")
    prompt = f"USER: <image>\n{question}{INSTR} ASSISTANT:"
    inp = lproc(text=prompt, images=img, return_tensors="pt").to(lmodel.device, torch.float16)
    with torch.no_grad():
        out = lmodel.generate(**inp, max_new_tokens=max_new_tokens, do_sample=False)
    return lproc.batch_decode(out[:, inp.input_ids.shape[1]:], skip_special_tokens=True)[0].strip()

if torch.cuda.is_available():
    df["llava"] = [llava_answer(r.image, r.question) for _, r in df.iterrows()]
    df["s_llava"] = [vu.score(r.kind, r.llava, r.answers, r.gold) for _, r in df.iterrows()]
    print(df.groupby("kind")[["s_small", "s_llava", "s_gemini"]].mean().round(3))

**What to expect on Colab (not measured here):** LLaVA-1.5 is strong on everyday photos, since VQAv2 is part of its training data, but its 336-pixel input makes small text in documents and charts hard to read. Published results for LLaVA-1.5-7B are around 78 percent on VQAv2 and far lower on document benchmarks. Expect the same shape as above: the gap to Gemini is smallest on photos and largest on documents. Treat that as a hypothesis to check with the scoring loop, not as a measured result.

## 9. Recap

- A VLM is a vision encoder, a connector and a language model; image tiles become tokens.
- Image splitting trades time for the ability to read fine text.
- A 256M model on CPU answers photos acceptably but struggles on charts and documents.
- Larger open models (LLaVA 7B) need a GPU and 4-bit quantization; hosted models need no hardware but send your images to a third party.

**Try yourself:** (1) change `max_new_tokens` to 4 and see which answers get cut; (2) try `HuggingFaceTB/SmolVLM-500M-Instruct`; (3) on Colab, run LLaVA and fill in the comparison table.

This completes Experiment 7.